In [1]:
import subprocess
import sys

# Function to install boto3
def install_boto3():
    try:
        # Check if boto3 is already installed
        import boto3
        print("boto3 is already installed")
    except ImportError:
        # If boto3 is not installed, install it using pip
        print("Installing boto3...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "boto3"])

# Call the function to install boto3
install_boto3()

boto3 is already installed


In [2]:
import boto3
from pyspark.sql import SparkSession

# Initialize Spark session
spark = SparkSession.builder \
    .appName("S3 File List and Size") \
    .getOrCreate()

# Initialize boto3 S3 client
s3_client = boto3.client('s3', region_name='ap-southeast-3')

# Define the bucket name
bucket_name = 'bucket-pkb'
folder_name = 'input'

# List all files in the bucket
response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=folder_name)

In [3]:
# Extract file details (key and size)
file_data = [(obj['Key'], obj['Size']) for obj in response.get('Contents', [])]

# Convert the file data to a PySpark DataFrame
df = spark.createDataFrame(file_data, ["FileName", "FileSize"])

# Show the DataFrame with file names and sizes
df.show(truncate=False)

+-------------------------------------------+----------+
|FileName                                   |FileSize  |
+-------------------------------------------+----------+
|input/                                     |0         |
|input/yelp_academic_dataset_review.json.zip|2220555375|
+-------------------------------------------+----------+



In [4]:
import os

# Get the current working directory
working_dir = os.getcwd()
print(f"Current working directory: {working_dir}")

Current working directory: /


In [5]:
import zipfile
import shutil

# Function to download and extract tar.gz or tgz files
def download_and_extract_tar(file_key):
    local_filename = file_key.split('/')[-1]  # Extract the file name from the S3 key
    download_path = f'/tmp/{local_filename}'  # Specify a local path (temporary folder)

    # Download the tar file from S3
    s3_client.download_file(bucket_name, file_key, download_path)
    print(f"Downloaded {file_key} to {download_path}")

    # Extract the tar.gz or tgz file
    if zipfile.is_zipfile(download_path):
        with zipfile.ZipFile(download_path, 'r') as zip_ref:
            extract_path = f"/tmp/{local_filename}_extracted"  # Directory to extract to
            os.makedirs(extract_path, exist_ok=True)
            zip_ref.extractall(extract_path)
            print(f"Extracted {local_filename} to {extract_path}")
            
            # List files in the extracted directory
            extracted_files = os.listdir(extract_path)
            print(f"Files in {extract_path}:")
            for file in extracted_files:
                print(file)
                
                # Upload each extracted file back to the S3 bucket
                # file_path = os.path.join(extract_path, file)
                # s3_upload_key = f'input/{file}'  # Define the S3 key (path in bucket) for the upload
                # s3_client.upload_file(file_path, bucket_name, s3_upload_key)
                # print(f"Uploaded {file_path} to s3://{bucket_name}/{s3_upload_key}")
                
                # Upload each extracted file to HDFS
                file_path = os.path.join(extract_path, file)
                hdfs_path = f"/user/hadoop/{file}"  # Define the HDFS path for the upload
                hdfs_put_command = f"hdfs dfs -put -f {file_path} {hdfs_path}"
                os.system(hdfs_put_command)  # Execute the HDFS put command
                print(f"Uploaded {file_path} to HDFS at {hdfs_path}")
            
            # Clean up extracted files and folder
            shutil.rmtree(extract_path)  # Remove the entire extracted folder
            print(f"Removed extracted folder: {extract_path}")
                
    else:
        print(f"{local_filename} is not a valid zip file")
    
    # Remove the downloaded tar.gz/tgz file
    if os.path.exists(download_path):
        os.remove(download_path)
        print(f"Removed downloaded file: {download_path}")
    
    # Delete the original tar.gz/tgz file from S3
    s3_client.delete_object(Bucket=bucket_name, Key=file_key)
    print(f"Deleted {file_key} from s3://{bucket_name}/{file_key}")

# Check if there are any contents in the folder
if 'Contents' in response:
    for obj in response['Contents']:
        file_key = obj['Key']
        # Check if the file ends with .tar.gz or .tgz
        if file_key.endswith('.zip'):
            download_and_extract_tar(file_key)
        else:
            print(f'Found {file_key}, but not a zip file')
else:
    print(f"No files found in folder: {folder_name}")

Found input/, but not a zip file
Downloaded input/yelp_academic_dataset_review.json.zip to /tmp/yelp_academic_dataset_review.json.zip
Extracted yelp_academic_dataset_review.json.zip to /tmp/yelp_academic_dataset_review.json.zip_extracted
Files in /tmp/yelp_academic_dataset_review.json.zip_extracted:
yelp_academic_dataset_review.json
Uploaded /tmp/yelp_academic_dataset_review.json.zip_extracted/yelp_academic_dataset_review.json to HDFS at /user/hadoop/yelp_academic_dataset_review.json
Removed extracted folder: /tmp/yelp_academic_dataset_review.json.zip_extracted
Removed downloaded file: /tmp/yelp_academic_dataset_review.json.zip
Deleted input/yelp_academic_dataset_review.json.zip from s3://bucket-pkb/input/yelp_academic_dataset_review.json.zip
